In [ ]:
import sys
import os 

# Get absolute path to project root
project_root = os.path.abspath("..")

# Add to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from utils import split_data, evaluate_model, plot_roc_curve, plot_confusion_matrix
from optuna.samplers import RandomSampler
from sklearn.preprocessing import (
    OneHotEncoder,  
    OrdinalEncoder, 
    PowerTransformer, 
    MinMaxScaler,
    RobustScaler
)

from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    balanced_accuracy_score, 
    f1_score, 
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)


## Load Data

In [ ]:
us_visa_data = pd.read_csv("EasyVisa.csv")
us_visa_data.head()

## Data Cleaning

During initial analysis we found that that the `case_id` has only unique values therefore high variance which is normal given that it's the unique identifier of each visa application however this feature is not useful for modeling so it can be safely removed. We also saw that sone of the records have negative values for the `no_of_employees` feature. It seems that this is a mistake during data collection since there is no pattern in the data that can suggest other reason. These records are very few so they can be safely removed too. 

In [ ]:
us_visa_data = us_visa_data.drop(columns=["case_id"])

negative_no_employees = us_visa_data[us_visa_data["no_of_employees"] < 0 ].index
us_visa_data = us_visa_data.drop(negative_no_employees)

us_visa_data.shape

## Feature Engineering

In [ ]:
from datetime import date

today_date = date.today()
current_year = today_date.year

# Creating a campany age feature to facilitate any future models
us_visa_data["company_age"] = current_year - us_visa_data["yr_of_estab"] 
us_visa_data = us_visa_data.drop(columns=["yr_of_estab"])

us_visa_data.shape

## Preprocessing

Before training any machine learning model the data has to be in a suitable format for this task. Machine learning models work only with numbers and don't know what to do with categories so categorical features have to be encoded in some way. Also some models don't work well with numerical features that are on vastly different scales and those features have to be rescaled to some range.

In [ ]:
numeric_features = [feature for feature in us_visa_data.columns if us_visa_data[feature].dtype != 'str']
ohe_features = [
    "continent", 
    "has_job_experience", 
    "requires_job_training", 
    "region_of_employment", 
    "unit_of_wage", 
    "full_time_position"
]

# The categories inside education_of_employee have an inherent order 
# (e.g. High School education is lower than Bachelor's degree etc.)
ordinal_features = ["education_of_employee"]
target_column = "case_status"
education_categories = ["High School", "Bachelor's", "Master's", "Doctorate"]

numeric_pipeline = Pipeline(steps=[
    # Makes skewed distributions more Gaussian-like. Many algorithms assume normally distributed data
    ("log_transform", PowerTransformer(standardize=False)),
    
    # Scales the data to be in the range between 0 and 1
    # Suitable for normally distributed data with no outliers 
    ("scale", MinMaxScaler())])

ohe_pipeline = Pipeline([
    # Creates a new binary column for each category
    ("ohe", OneHotEncoder(sparse_output=False, drop="first"))
])

ordinal_pipeline = Pipeline(steps=[
    ("encode", OrdinalEncoder(categories=[education_categories]))
]) 

transformer = ColumnTransformer([
    ("numeric_pipeline", numeric_pipeline, numeric_features),
    ("ohe_pipeline", ohe_pipeline, ohe_features),
    ("ordinal_pipeline", ordinal_pipeline, ordinal_features)
])

# Encoding the target column as binary labels (0 -> Denied, 1 -> Certified)
us_visa_data["case_status"] = us_visa_data["case_status"].apply(lambda row: 0 if row == "Denied" else 1)

In [ ]:
us_visa_data["case_status"]

## Model Building

### Train, Validation and Test Splits

In this step the data is split into the training, validation and test sets. The training set is used to train models to learn from data, the validation set is to compare the performance of different models and evaluation after training and after hyperparameter tuning and select the best model for final evaluation.

The test set remains hidden from the model during training and evaluation and is used for assesing modle generalization on new unseen data. A good rule of thumb is that the test should be 20% of the whole data set but this number can vary. But here the more important idea is that the test set should be small enough to leave enough records for training and validation but not too small since it could be unstable and not repesentative of the entire dataset. Cross-validation is not needed here since there are enough samples in the dataset.

In [ ]:
visa_attributes = us_visa_data.drop(columns=target_column) # Input features
visa_target = us_visa_data[target_column] # Target column

attributes_train_and_val, attributes_test, target_train_and_val, target_test = split_data(
    input_features=visa_attributes,
    target_labels=visa_target,
    test_size=0.2
)

attributes_train, attributes_val, target_train, target_val = split_data(
    input_features=attributes_train_and_val,
    target_labels=target_train_and_val,
    test_size=4000
)

print(attributes_train_and_val.shape, attributes_test.shape, target_train_and_val.shape, target_test.shape)
print(attributes_train.shape, attributes_val.shape, target_train.shape, target_val.shape)

In [ ]:
test_set = pd.concat([attributes_test, target_test], axis=1)
test_set.to_csv("test.csv", index=False)

del test_set
del attributes_test
del target_test

### Evaluation Metrics

Given the imbalanced dataset some suitable metrics for this case are __roc_auc_score__,  __balanced accuracy__ and __f1_score__

- **balanced accuracy**: the average recall for each class. This is our main metric
- **f1_score**: the harmonic mean of precision and recall. It is suitable for cases where precision and recall are of equal importance
- **roc_auc_score**: the area under the Receiver Operating Characteristic (ROC) curve. This metric compares the true positive and false positive rate at various classification thresholds.

### Dummy Model

The first model is a simple dummy model that predicts the most common class. It has high bias as expected since it doesn't learn anything interesting from the data besides the most common label. It achived a balanced accuracy score of 50% and  a f1 score of 80%.

In [ ]:
base_pipeline = Pipeline([
    ("transform", transformer),
    # A simple model with no learning capacity to serve as baseline for later comparison 
    ("model", DummyClassifier(strategy="most_frequent"))
])

base_pipeline.fit(attributes_train, target_train)

In [ ]:
evaluate_model(base_pipeline, attributes_train, target_train)

In [ ]:
plot_roc_curve(
    model=base_pipeline, 
    input_features=attributes_train, 
    target_labels=target_train,
    plot_title="ROC Curve for Dummy Classifier"
)

### Linear Support Vector Classifier

The next model is a Support Vector Classifier. It performs better than the baseline but also seems to have high bias. The results for both training and validation sets and both metrics are very similar which indicates that probably the default regulariation strenght is too high.

In [ ]:
svm_pipeline = Pipeline([
    ("transform", transformer),
    # A more complex linear model with a good performance both for categorical and numeric features
    ("model", LinearSVC(random_state=42))
])

svm_pipeline.fit(attributes_train, target_train)

In [ ]:
train_accuracy = balanced_accuracy_score(target_train, svm_pipeline.predict(attributes_train))
train_f1_score = f1_score(target_train, svm_pipeline.predict(attributes_train))
train_roc_auc = roc_auc_score(target_train, svm_pipeline.decision_function(attributes_train))

print(f"Train set balanced accuracy score: {train_accuracy:.4f}")
print(f"Train set f1 score: {train_f1_score:.4f}")
print(f"Train set ROC AUC score: {train_roc_auc:.4f}")

In [ ]:
val_accuracy = balanced_accuracy_score(target_val, svm_pipeline.predict(attributes_val))
val_f1_score = f1_score(target_val, svm_pipeline.predict(attributes_val))
val_roc_auc_score = roc_auc_score(target_val, svm_pipeline.decision_function(attributes_val))

print(f"Validation set balanced accuracy score: {val_accuracy:.4f}")
print(f"Validation set f1 score: {val_f1_score:.4f}")
print(f"Validation set ROC AUC score: {val_roc_auc_score:.4f}")

In [ ]:
plot_roc_curve(
    model=svm_pipeline,
    input_features=attributes_val,
    target_labels=target_val,
    plot_title="ROC Curve for LinearSVC"
)

### Decision Tree Classifier

The next model is a DecisionTreeClassifier. This model perform better than the LinearSVC model but has high variance and overfits on the training data and doesn't perform well on the validation data. This happens because an unregularized tree can grow infinately deep untill the leaf nodes at the bottom level contain only pure samples and  absolutely perfect classification. 

In [ ]:
tree_pipeline = Pipeline([
    ("transform", transformer),
    # A more complex model with a good performance on data with mainly categorical features.
    # Most probably will overfit the training data
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_pipeline.fit(attributes_train, target_train)

In [ ]:
evaluate_model(tree_pipeline, attributes_train, target_train)

In [ ]:
evaluate_model(tree_pipeline, attributes_val, target_val)

In [ ]:
plot_roc_curve(
    model=tree_pipeline,
    input_features=attributes_val,
    target_labels=target_val,
    plot_title="ROC Curve for DecisionTreeClassifier"
)

### Random Forest Classifier

The next model is a Random Forest Classifier. It also has high variance and overfits to the training data and doesn't perform well on the validation data. 

In [ ]:
forest_pipeline = Pipeline([
    ("transform", transformer),
    
    # An ensamble model made of multiple decision trees trained on different subsets of the training data
    # Has a lower chance of overfitting to the training data.
    ("model", RandomForestClassifier(random_state=42))
])

forest_pipeline.fit(attributes_train, target_train)

In [ ]:
evaluate_model(forest_pipeline, attributes_train, target_train)

In [ ]:
evaluate_model(forest_pipeline, attributes_val, target_val)

In [ ]:
plot_roc_curve(
    model=forest_pipeline, 
    input_features=attributes_val,
    target_labels=target_val,
    plot_title="ROC Curve for RandomForestClassifier"
)

## Hyperparameter Tuning

Let's try to squeeze out some more performance from the model using hyperparameter tuning. For this task the `optuna` package is used instead of scikit-learn's methods like `GridSearchCV` and `RandomizedSearchCV` since it provides more efficent algorithms for hyperparameter tuning.

Even though `optuna` is better optimized hyperparameter tuning still takes time so only RandomForestClassifer and LinearSVC models are tuned. The other models are skipped since the dummy model is only used as baseline for comparison and doesn't actually learn anything form the data and it doesn't have any hyperparameters; the DecisionTreeClassifer has similar hyperprameters as the RandomForestClassifier and their effect is the same.

We are optimizing for the f1_score since it gives equal imprtnace to false negatives and false positives.

In [ ]:
def objective(trial):
    classifier_name = trial.suggest_categorical("classifer", ["LinearSVC", "RandomForest"])
    classifier_obj = None

    # Hyperparameters to optimize
    if classifier_name == "RandomForest":
        # Controls the number of decision trees in the ensamble. 
        # Smaller number leads to simplier model with less chance of overfitting
        n_estimators = trial.suggest_int("n_estimators", 50, 500)
    
        # Controls the depth of each tree in the ensamble
        max_depth = trial.suggest_int("max_depth", 2, 10)
    
        # Controls the minimum number of samples required to split an internal node
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    
        # Controls the minimum number of samples required to be at a leaf node
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
        classifier_obj = Pipeline([
            ("transform", transformer),
            ("model", RandomForestClassifier(
                n_estimators=n_estimators, 
                max_depth=max_depth, 
                min_samples_split=min_samples_split, 
                min_samples_leaf=min_samples_leaf,
                random_state=42
            ))
        ])

    else:
        # Controls the regularization strength of the model.
        # Smaller number means stronger regularization and risk of underfitting, 
        # larger number means weaker regularization and risk of overfitting
        svm_c = trial.suggest_float('svm_c', 1e-10, 1e10, log=True)
        classifier_obj = Pipeline([
            ("transform", transformer),
            ("model", LinearSVC(C=svm_c, random_state=42))
        ])

    classifier_obj.fit(attributes_train, target_train)
    predictions = classifier_obj.predict(attributes_val)

    score = f1_score(target_val, predictions)
    return score
        

In [ ]:
# # Cerate a study object
# study = optuna.create_study(direction="maximize", sampler=RandomSampler(seed=42))

# # Begin optimization process
# study.optimize(objective, n_trials=100)

In [ ]:
# best_params = study.best_params
# best_score = study.best_value

# print(f"Best hyperparameters: {best_params}")
# print(f"Best score: {best_score}")

## Final Evaluation

The best performing model seems to be the Random Forest Classifer. It achived a 66% balanced accuracy score, 83% f1 score and 79% roc auc score on the test set. The results are very simmilar to the ones on the training set but are still somewhat low 

In [ ]:
us_visa_data_test = pd.read_csv("test.csv")
print(us_visa_data_test)

In [ ]:
attributes_test = us_visa_data_test.drop(columns=[target_column])
target_test = us_visa_data_test[target_column]

print(attributes_test.head())
print(target_test.head())

In [ ]:
best_forest_pipeline = Pipeline([
    ("transform", transformer),
    ("model", RandomForestClassifier(
        n_estimators=363,
        max_depth=7,
        min_samples_leaf=7,
        min_samples_split=2,
        random_state=42
    ))
])

best_forest_pipeline.fit(attributes_train, target_train)

In [ ]:
evaluate_model(best_forest_pipeline, attributes_train, target_train)

In [ ]:
evaluate_model(best_forest_pipeline, attributes_test, target_test)

In [ ]:
plot_roc_curve(
    model=best_forest_pipeline,
    input_features=attributes_test,
    target_labels=target_test,
    plot_title="ROC Curve for Tuned RandomForestClassifier"
)

As seen form the confusion matrix bellow the amount of false positives is high. The could be resolved with a different classification threshold

In [ ]:
plot_confusion_matrix(
    model=best_forest_pipeline,
    target_labels=target_test,
    input_features=attributes_test,
    plot_title="Confusion Matrix for Tuned RandomForestClassifier"
)

## Classification Thereshold Tuning

The default classification threshold for scikit-learn estimators is 0.5, however this is rarely useful in cases of class imbalance where different misclassification errors have different costs. The optimal threshold taking into account we're optimizing for f1 score is 0.503. This means that if an appilcation has greater than 50.3% probability of being approved it is consdered approved by the classifer

In [ ]:
thresholded_classifier = TunedThresholdClassifierCV(
    estimator=best_forest_pipeline,
    scoring="f1",
    random_state=42
)

thresholded_classifier.fit(attributes_train, target_train)

In [ ]:
evaluate_model(thresholded_classifier, attributes_train, target_train)

In [ ]:
evaluate_model(thresholded_classifier, attributes_test, target_test)
print(f"New optimal threshold: {thresholded_classifier.best_threshold_:.4f}")

In [ ]:
plot_roc_curve(
    model=thresholded_classifier,
    input_features=attributes_test,
    target_labels=target_test,
    plot_title="ROC Curve for Thersholded RandomForestClassifier"
)

With the new classification threshold, the number of false negatives dropped from 990 to 966, however the number of false negatives increased from 293 to 312. The number of true negatives also increased from 699 to 723.

In [ ]:
plot_confusion_matrix(
    model=thresholded_classifier,
    input_features=attributes_test,
    target_labels=target_test,
    plot_title="Confusion Matrix After Thershold Tuning"
)

## Conclusion

This project attempted to solve the classification problem of classifying visa applications as cetified or denied.
Mutiple machine algorithm were tested but the best performing one proved to be the Random Forest Classifer. Some other options were:

- **Support Vector Classifer**: this model had high bias and coludn't learn the underling patterns in the data. We assumed that the underfitting problem was due to the default regularization strength beeing to high. We tuned the `C` hyperparameter of the model together with the Random Forest Classifer and the Random Forest Classifer proved to be the better choice.
- **Decision Tree Classifer**: this model had high variance and was overfitting on the training data. We slipped tuning it since it has the same hyperparameters as the Random Forest Classifer and they have the same effect.

### Next Steps
Some steps that could be taken to improve model performance could be:
- Collecing more data
- Creating more meaningful features with more careful feature engineering
- A better preprocessing pipeline